# Drone Trajectory Planner

In this project, we will develop the drone trajectory planner. This notebook serves as the main file for the project, where we will refer to the instructions and demonstrate our code.

Please follow week by week instructions, which includes writing the code in the `src/` folder.

In [1]:
# Import all the files and libraries required for the project
%load_ext autoreload
%autoreload 2
import copy
import numpy as np

from src.camera_utils import compute_image_footprint_on_surface, compute_image_footprint_non_nadir, compute_ground_sampling_distance, project_world_point_to_image
from src.data_model import Camera, DatasetSpec
from src.plan_computation import compute_distance_between_images, compute_speed_during_photo_capture, generate_photo_plan_on_grid
from src.visualization import (
    plot_photo_plan,
    plot_3d_trajectory,
    plot_footprint_overlay,
    plot_parameter_sensitivity,
    plot_overlap_sidelap_heatmap,
    plot_scenario_comparison,
    plot_angle_analysis,
    plot_efficiency_gauge,
)

import plotly.io as pio
pio.renderers.default = 'iframe'

# Week 1: Introduction

No code contribution expected this week

# Week 2: Implement data models for dataset specification and camera

For this week, we will model the dataset specification.

- Overlap: the ratio (in 0 to 1) of scene shared between two consecutive images.
- Sidelap: the ratio (in 0 to 1) of scene shared between two images in adjacent rows.
- Height: the height of the scan above the ground (in meters).
- Scan_dimension_x: the horizontal size of the rectangle to be scanned (in meters).
- Scan_dimension_y: the vertical size of the rectangle to be scanned (in meters).
- exposure_time_ms: the exposure time for each image (in milliseconds).


$\color{red}{\text{TODO: }}$ Implement `DatasetSpec` in `src/data_model.py`


In [2]:
# Model the nomimal dataset spec

overlap = 0.7
sidelap = 0.7
height = 30.48 # 100 ft
scan_dimension_x = 150
scan_dimension_y = 150
exposure_time_ms = 2 # 1/500 exposure time

dataset_spec = DatasetSpec(overlap, sidelap, height, scan_dimension_x, scan_dimension_y, exposure_time_ms)

print(f"Nominal specs: {dataset_spec}")

Nominal specs: DatasetSpec(overlap=0.7, sidelap=0.7, height=30.48m, scan=(150x150), exposure=2ms, camera_angle=90.0°)


## Model the camera parameters

We want to model the following camera parameters in Python:
- focal length along x axis (in pixels)
- focal length along y axis (in pixels)
- Size of the sensor along the x axis (in mm)
- Size of the sensor along the y axis (in mm)
- Number of pixels in the image along the x axis
- Number of pixels in the image along the y axis

I recommend to use `dataclasses` ([Python documentation](https://docs.python.org/3/library/dataclasses.html), [Blog](https://www.dataquest.io/blog/how-to-use-python-data-classes/) to model these parameters.

$\color{red}{\text{TODO: }}$ Implement `Camera` in `src/data_model.py`

In [3]:
# Define the parameters for Skydio VT300L - Wide camera
# Ref: https://support.skydio.com/hc/en-us/articles/20866347470491-Skydio-X10-camera-and-metadata-overview
fx = 4938.56 # px
fy = 4936.49 # px
sensor_size_x_mm = 13.107 # single pixel size * number of pixels in X dimension
sensor_size_y_mm = 9.830 # single pixel size * number of pixels in Y dimension
num_pixels_x = 8192
num_pixels_y = 6144

camera_x10 = Camera(fx, fy, sensor_size_x_mm, sensor_size_y_mm, num_pixels_x, num_pixels_y)

In [4]:
print(f"X10 camera model: {camera_x10}")

X10 camera model: Camera(fx=4938.56, fy=4936.49, sensor_size_x_mm=13.107, sensor_size_y_mm=9.83, num_pixels_x=8192, num_pixels_y=6144)


# Week 3: Camera Operations

We plan to write utility functions to
- project a 3D world point to an image
- Compute image footprint on a surface
- Compute the Ground Sampling Distance

## Project 3D world points into the image coordinates


![Camera Projection](assets/image_projection.png)
Reference: [Robert Collins CSE483](https://www.cse.psu.edu/~rtc12/CSE486/lecture12.pdf)


Equations to implement:
$$ x = f_x \frac{X}{Z} $$
$$ y = f_y \frac{Y}{Z} $$

$\color{red}{\text{TODO: }}$ Implement function `project_world_point_to_image` in `src/camera_utils.py`

In [5]:
point_3d = (25, -30, 50)
expected_xy = (2469.28, -2961.894)
xy = project_world_point_to_image(camera_x10, point_3d)

print(f"{point_3d} projected to {xy}")

assert np.allclose(xy, expected_xy, atol=1e-2)

(25, -30, 50) projected to (2469.28, -2961.894)


## Compute Image Footprint on the surface

We have written code to *project* a 3D point into the image. The reverse operation is reprojection, where we take $(x, y)$ and compute the $(X, Y)$ for a given value of $Z$. Note that while going from 3D to 2D, the depth becomes ambiguous so we need the to specify the $Z$.

An image's footprint is the area on the surface which is captured by the image. We can take the two corners of the image and reproject them at a given distance to obtain the width and length of the image.

$\color{red}{\text{TODO: }}$ Implement function `compute_image_footprint_on_surface` in `src/camera_utils.py`

In [6]:
footprint_at_100m = compute_image_footprint_on_surface(camera_x10, 100)
expected_footprint_at_100m = (165.88, 124.46)

print(f"Footprint at 100m = {footprint_at_100m}")

assert np.allclose(footprint_at_100m, expected_footprint_at_100m, atol=1e-2)


Footprint at 100m = (165.8783127065379, 124.46090238205689)


In [7]:
footprint_at_200m = compute_image_footprint_on_surface(camera_x10, 200)
expected_footprint_at_200m = (165.88*2, 124.46*2)

print(f"Footprint at 200m = {footprint_at_200m}")

assert np.allclose(footprint_at_200m, expected_footprint_at_200m, atol=1e-2)

Footprint at 200m = (331.7566254130758, 248.92180476411377)


## Ground Sampling Distance

Ground sampling distance is the length of the ground (in m) captured by a single pixel. We have the image footpring (the dimensions of ground captured by the whole sensor, and the number of pixels along the horizontal and vertical dimension. Can we get GSD from these two quantities?

Note: Please return just one value of the GSD. Take the mininum of the values along the two axes.

In [8]:
gsd_at_100m = compute_ground_sampling_distance(camera_x10, 100)
expected_gsd_at_100m = 0.0202

print(f"GSD at 100m: {gsd_at_100m}")

assert np.allclose(gsd_at_100m, expected_gsd_at_100m, atol=1e-4)

GSD at 100m: 0.020248817469059804


## Bonus: Reprojection from 2D to 3D

If we have a 2d pixel location of a point along with the camera model, can we go back to 3D?
Do we need any additional information.


$\color{red}{\text{TODO: }}$ Implement function `reproject_image_point_to_world` in `src/camera_utils.py` and demonstrate it by running it in the notebook. Confirm that your reprojection + projection function are by projecting a 3d point to image, and reprojecting it back to 3D.

# Week 4: Compute Distance Between Photos

The overlap and sidelap are the ratio of the dimensions shared between two photos. We already know the footprint of a single image at a given distance. Can we convert the ratio into actual distances? And how does the distance on the surface relate to distance travelled by the camera?

$\color{red}{\text{TODO: }}$ Implement `compute_distance_between_images` in `src/plan_computation.py`



In [9]:
computed_distances = compute_distance_between_images(camera_x10, dataset_spec)
expected_distances = np.array([15.17, 11.38], dtype=np.float32)

print(f"Computed distance for X10 camera with nominal dataset specs: {computed_distances}")

assert np.allclose(computed_distances, expected_distances, atol=1e-2)

Computed distance for X10 camera with nominal dataset specs: (15.16791291388583, 11.380704913815284)


$\color{red}{\text{TODO: }}$ define >=2 more specifications/camera parameters and check the computed distances. Does that align with your expections


In [10]:
camera_ = copy.copy(camera_x10)
dataset_spec_ = copy.copy(dataset_spec)

computed_distances_ = compute_distance_between_images(camera_, dataset_spec_)
print(f"Computed distance: {computed_distances_}")

Computed distance: (15.16791291388583, 11.380704913815284)


## Bonus: Non-Nadir photos

We have solved for the distance assuming that the camera is facing straight down to the ground. This is called [Nadir scanning](https://support.esri.com/en-us/gis-dictionary/nadir). However, in practise we might want a custom gimbal angle.

Your bonus task is to make the distance computation general. Introduce a double `camera_angle` parameter (which is the angle from the X-axis) in the dataset specification, and work out how to adapt your computation. Feel free to reach out to Ayush to discuss ideas and assumptions!

![Non Nadir Footprint](assets/non_nadir_gimbal_angle.png)

In [11]:
# Bonus: Non-Nadir Distance Computation
# When the camera is tilted (non-nadir), the ground footprint changes.
# We added a camera_angle parameter to DatasetSpec (angle from horizontal, 90 = nadir).

from src.camera_utils import compute_image_footprint_non_nadir

# Compare nadir vs non-nadir footprint at 90 degrees (should be identical)
fp_nadir = compute_image_footprint_on_surface(camera_x10, dataset_spec.height)
fp_at_90 = compute_image_footprint_non_nadir(camera_x10, dataset_spec.height, 90.0)

print(f"Nadir footprint:      ({fp_nadir[0]:.4f}m, {fp_nadir[1]:.4f}m)")
print(f"Non-nadir at 90°:     ({fp_at_90[0]:.4f}m, {fp_at_90[1]:.4f}m)")

Nadir footprint:      (50.5597m, 37.9357m)
Non-nadir at 90°:     (50.5597m, 37.9357m)


In [12]:
# Compare footprints and distances at various camera angles
print(f"{'Angle':>6} | {'Footprint X':>12} | {'Footprint Y':>12} | {'Distance X':>12} | {'Distance Y':>12}")
print("-" * 72)

for angle in [90, 80, 70, 60, 50, 45]:
    spec = DatasetSpec(0.7, 0.7, 30.48, 150, 150, 2, camera_angle=angle)
    d = compute_distance_between_images(camera_x10, spec)
    fp = compute_image_footprint_non_nadir(camera_x10, 30.48, angle)
    print(f"{angle:>5}° | {fp[0]:>10.2f}m | {fp[1]:>10.2f}m | {d[0]:>10.2f}m | {d[1]:>10.2f}m")

 Angle |  Footprint X |  Footprint Y |   Distance X |   Distance Y
------------------------------------------------------------------------
   90° |      50.56m |      37.94m |      15.17m |      11.38m
   80° |      53.27m |      38.52m |      15.98m |      11.56m
   70° |      63.00m |      40.37m |      18.90m |      12.11m
   60° |      87.47m |      43.80m |      26.24m |      13.14m
   50° |     167.08m |      49.52m |      50.12m |      14.86m
   45° |     323.99m |      53.65m |      97.20m |      16.09m


In [13]:
# Compare nadir vs tilted distance computation
spec_nadir = DatasetSpec(0.7, 0.7, 30.48, 150, 150, 2)
spec_60 = DatasetSpec(0.7, 0.7, 30.48, 150, 150, 2, camera_angle=60.0)
spec_45 = DatasetSpec(0.7, 0.7, 30.48, 150, 150, 2, camera_angle=45.0)

dist_nadir = compute_distance_between_images(camera_x10, spec_nadir)
dist_60 = compute_distance_between_images(camera_x10, spec_60)
dist_45 = compute_distance_between_images(camera_x10, spec_45)

print(f"Nadir (90°) distances: ({dist_nadir[0]:.2f}m, {dist_nadir[1]:.2f}m)")
print(f"Tilted (60°) distances: ({dist_60[0]:.2f}m, {dist_60[1]:.2f}m)")
print(f"Tilted (45°) distances: ({dist_45[0]:.2f}m, {dist_45[1]:.2f}m)")

Nadir (90°) distances: (15.17m, 11.38m)
Tilted (60°) distances: (26.24m, 13.14m)
Tilted (45°) distances: (97.20m, 16.09m)


# Week 5: Compute Maximum Speed For Blur Free Photos

To restrict motion blur due to camera movement to tolerable limits, we need to restrict the speed such that the image contents move less than 1px away. 

How much does 1px of movement translate to movement of the scene on the ground? It is the ground sampling distance!
From previous week, we know that this is the maximum movement the camera can have. 
We have the distance now. To get speed we need to divide it with time. Do we have time already in our data models?

$\color{red}{\text{TODO: }}$ Implement `compute_speed_during_photo_capture` in `src/plan_computation.py`.

In [14]:
computed_speed = compute_speed_during_photo_capture(camera_x10, dataset_spec, allowed_movement_px=1)
expected_speed = 3.09

print(f"Computed speed during photo captures: {computed_speed:.2f}")

assert np.allclose(computed_speed, expected_speed, atol=1e-2)

Computed speed during photo captures: 3.09


$\color{red}{\text{TODO: }}$ define >= 2more specifications/camera parameters and check the computed distances. Does that align with your expectations


In [15]:
camera_ = copy.copy(camera_x10)
dataset_spec_ = copy.copy(dataset_spec)

computed_speed_ = compute_speed_during_photo_capture(camera_, dataset_spec_)
print(f"Computed distance: {computed_speed_:.2f}")

Computed distance: 3.09


# Week 6: Generate Full Flight Plans  

We now have all the tools to generate the full flight plan.

Steps for this week:
1. Define the `Waypoint` data model. What attributes should the data model have?
   1. For Nadir scans, just the position of the camera is enough as we will always look drown to the ground.
   2. For general case (bonus), we also need to define where the drone will look at.
3. Implement the function `generate_photo_plan_on_grid` to generate the full plan.
   1. Compute the maximum distance between two images, horizontally and vertically.
   2. Layer the images such that we cover the whole scan area. Note that you need to take care when the scan dimension is not a multiple of distance between images. Example: to cover 45m length with 10m between images, we would need 4.5 images. Not possible. 4 images would not satisfy the overlap, so we should go with 5. How should we arrange 5 images in the given 45m.
   3. Assign the speed to each waypoint.

$\color{red}{\text{TODO: }}$ Implement:
- `Waypoint` in `src/data_model.py`
- `generate_photo_plan_on_grid` in `src/plan_computation.py`.

In [16]:
computed_plan = generate_photo_plan_on_grid(camera_x10, dataset_spec) 

print(f"Computed plan with {len(computed_plan)} waypoints")

Computed plan with 140 waypoints


In [17]:
MAX_NUM_WAYPOINTS_TO_PRINT = 20

for idx, waypoint in enumerate(computed_plan[:MAX_NUM_WAYPOINTS_TO_PRINT]):
    print(f"Idx {idx}: {waypoint}")
if len(computed_plan) >= MAX_NUM_WAYPOINTS_TO_PRINT:
    print("...")

Idx 0: Waypoint(x=0.0, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 1: Waypoint(x=16.666666666666668, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 2: Waypoint(x=33.333333333333336, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 3: Waypoint(x=50.0, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 4: Waypoint(x=66.66666666666667, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 5: Waypoint(x=83.33333333333334, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 6: Waypoint(x=100.0, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 7: Waypoint(x=116.66666666666667, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 8: Waypoint(x=133.33333333333334, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 9: Waypoint(x=150.0, y=0.0, z=30.48, speed=3.0859197822847144)
Idx 10: Waypoint(x=150.0, y=11.538461538461538, z=30.48, speed=3.0859197822847144)
Idx 11: Waypoint(x=133.33333333333334, y=11.538461538461538, z=30.48, speed=3.0859197822847144)
Idx 12: Waypoint(x=116.66666666666667, y=11.538461538461538, z=30.48, speed=3.

## Bonus: Time computation 

if you have some time, you can implement a time computation function. We can make the drone fly as fast as possible between photos, but make sure it can decelerate back to the required speed at the photos. Please use the following data: 
- Max drone speed: 16m/s.
- Max acceleration: 3.5 m/s^2.

Hint: you might need to use a trapezoidal/triangular speed profile

# Week 7: Visualize Flight Plans

This week, we will use a third party plotting framework called [Plotly](https://plotly.com/python/) to visualize our plans. Please follow this [tutorial](https://www.kaggle.com/code/kanncaa1/plotly-tutorial-for-beginners) to gain some basic experience with Plotly, and then come up with your own visualization function. You are free to choose to come up with your own visualization, and use something other than Plotly.

$\color{red}{\text{TODO: }}$ Implement `plot_photo_plan` in `src/visualization.py`

In [18]:
fig = plot_photo_plan(computed_plan)
fig.show()

$\color{red}{\text{TODO: }}$ Perform the following experiments (and any other you can think of) where we change just one parameter of the input camera/dataset specification and observe the change in the output plan. 

1. Change overlap and confirm it affects the consecutive images
2. Change sidelap and confirm it does not affect the consecutive images
3. Change the height of the scan and document the affect on scan plans
4. Change exposure time

Each experiment should specify: 
1. Input params you are changing
2. Impact you observe
3. explanation behind the change in output (intuition based or a text explanation is preffered over using equations)
4. Practical implication of the correlation: how can I drone pilot use this result

In [19]:
camera_ = copy.deepcopy(camera_x10)
dataset_spec_ = copy.deepcopy(dataset_spec)

dataset_spec_.exposure_time_ms = 1000

print(camera_, dataset_spec_)

fig = plot_photo_plan(generate_photo_plan_on_grid(camera_, dataset_spec_))
fig.show()

Camera(fx=4938.56, fy=4936.49, sensor_size_x_mm=13.107, sensor_size_y_mm=9.83, num_pixels_x=8192, num_pixels_y=6144) DatasetSpec(overlap=0.7, sidelap=0.7, height=30.48m, scan=(150x150), exposure=1000ms, camera_angle=90.0°)


### Experiment 1: Effect of Overlap on Flight Plan

**Parameter changed:** Overlap from 0.7 (70%) to 0.5 (50%)

**Expected impact:** Lower overlap means the drone covers more new ground per photo, so fewer waypoints are needed along X.

**Practical implication:** A drone pilot can lower overlap for faster area coverage when ultra-precise photogrammetry isn't needed (e.g. visual inspection). Higher overlap is essential for 3D reconstruction tasks.

In [20]:
# ── Experiment 1: Effect of Overlap on the Flight Plan ──
# We reduce overlap from 70% to 50% while keeping all other params constant.
# Overlap controls how much consecutive images share along the flight direction (X).
# Lower overlap → larger spacing between photos → fewer waypoints per row.
# The sidelap (row spacing) remains the same, so the number of rows is unchanged.

spec_low_ol = DatasetSpec(0.5, 0.7, 30.48, 150, 150, 2)
plan_low_ol = generate_photo_plan_on_grid(camera_x10, spec_low_ol)

# Compare waypoint counts: fewer waypoints confirms that overlap only affects
# the along-track (X) spacing, not the cross-track (Y) row layout.
print(f"Nominal (70% overlap): {len(computed_plan)} waypoints")
print(f"Low overlap (50%):     {len(plan_low_ol)} waypoints")

fig = plot_photo_plan(plan_low_ol, title='🛩️ Experiment 1: 50% Overlap')
fig.show()

Nominal (70% overlap): 140 waypoints
Low overlap (50%):     84 waypoints


### Experiment 2: Effect of Sidelap on Flight Plan

**Parameter changed:** Sidelap from 0.7 (70%) to 0.5 (50%)

**Expected impact:** Lower sidelap reduces the number of rows (Y direction), but does NOT affect consecutive images within a row.

**Practical implication:** Reducing sidelap saves flight time by cutting rows but may create gaps in cross-track coverage.

In [21]:
# ── Experiment 2: Effect of Sidelap on the Flight Plan ──
# We reduce sidelap from 70% to 50% while keeping all other params constant.
# Sidelap controls how much adjacent rows share in the Y direction.
# Lower sidelap → larger vertical spacing → fewer rows needed to cover the area.
# Crucially, sidelap does NOT affect the spacing within a single row (overlap does that).

spec_low_sl = DatasetSpec(0.7, 0.5, 30.48, 150, 150, 2)
plan_low_sl = generate_photo_plan_on_grid(camera_x10, spec_low_sl)

# Compare: the per-row count stays the same, but fewer rows means fewer total waypoints.
print(f"Nominal (70% sidelap): {len(computed_plan)} waypoints")
print(f"Low sidelap (50%):     {len(plan_low_sl)} waypoints")

fig = plot_photo_plan(plan_low_sl, title='🛩️ Experiment 2: 50% Sidelap')
fig.show()

Nominal (70% sidelap): 140 waypoints
Low sidelap (50%):     80 waypoints


### Experiment 3: Effect of Height on Flight Plan

**Parameter changed:** Height from 30.48m to 60m (doubled)

**Expected impact:** Doubling height doubles the footprint, so fewer images are needed. However, GSD worsens (less detail).

**Practical implication:** Pilots should fly higher for rapid area surveys and lower for high-detail inspections. The trade-off is GSD quality vs. flight time.

In [22]:
# ── Experiment 3: Effect of Flight Height on the Plan ──
# We double the height from 30.48m to 60m while keeping overlap/sidelap fixed.
# Higher altitude → each image covers a larger ground footprint (footprint ∝ height).
# Since the footprint is bigger, the required spacing between photos also increases,
# meaning fewer photos are needed to cover the same 150×150m area.
# Trade-off: GSD (resolution) worsens linearly with height (GSD = height / focal_length).

spec_high_h = DatasetSpec(0.7, 0.7, 60.0, 150, 150, 2)
plan_high_h = generate_photo_plan_on_grid(camera_x10, spec_high_h)

# Compute GSD at both heights to quantify the resolution trade-off
gsd_nom = compute_ground_sampling_distance(camera_x10, 30.48)
gsd_high = compute_ground_sampling_distance(camera_x10, 60.0)

# Result: ~4x fewer waypoints but GSD roughly doubles (less detail per pixel)
print(f"Nominal (30.48m): {len(computed_plan)} waypoints, GSD={gsd_nom*100:.2f} cm/px")
print(f"High alt (60m):   {len(plan_high_h)} waypoints, GSD={gsd_high*100:.2f} cm/px")

fig = plot_photo_plan(plan_high_h, title='🛩️ Experiment 3: 60m Height')
fig.show()

Nominal (30.48m): 140 waypoints, GSD=0.62 cm/px
High alt (60m):   42 waypoints, GSD=1.21 cm/px


### Experiment 4: Effect of Exposure Time on Speed

**Parameter changed:** Exposure time from 2ms to 10ms

**Expected impact:** Longer exposure means the drone must fly slower to avoid motion blur. The waypoint count stays the same but speed decreases 5x.

**Practical implication:** In low-light conditions requiring longer exposures, pilots must accept much slower flight speeds, dramatically increasing mission time.

In [23]:
# ── Experiment 4: Effect of Exposure Time on Flight Speed ──
# We increase exposure from 2ms to 10ms (5× longer shutter open).
# Exposure time does NOT affect footprint or waypoint count—those depend on
# camera geometry and overlap/sidelap only.
# However, longer exposure means the drone must fly slower to keep motion blur
# under 1 pixel. Speed = GSD / exposure_time, so 5× longer exposure → 5× slower.
# This is critical in low-light conditions where pilots need longer exposures.

spec_slow_exp = DatasetSpec(0.7, 0.7, 30.48, 150, 150, 10)
plan_slow_exp = generate_photo_plan_on_grid(camera_x10, spec_slow_exp)

# Compute max allowable speed at both exposure times
speed_nom = compute_speed_during_photo_capture(camera_x10, dataset_spec)
speed_slow = compute_speed_during_photo_capture(camera_x10, spec_slow_exp)

# Result: same waypoints (geometry unchanged), but speed drops from ~3.09 to ~0.62 m/s
# This means the mission takes ~5× longer even though the flight path is identical.
print(f"Nominal (2ms):     speed={speed_nom:.2f} m/s, {len(computed_plan)} waypoints")
print(f"Slow exp (10ms):   speed={speed_slow:.2f} m/s, {len(plan_slow_exp)} waypoints")

fig = plot_photo_plan(plan_slow_exp, title='🛩️ Experiment 4: 10ms Exposure')
fig.show()

Nominal (2ms):     speed=3.09 m/s, 140 waypoints
Slow exp (10ms):   speed=0.62 m/s, 140 waypoints


---

# Week 7: Comprehensive Visualizations

Below we demonstrate 7 advanced visualizations of our flight plans using the enhanced `src/visualization.py` module. Each visualization provides unique insights into the drone trajectory planning process.

## 1. 3D Flight Trajectory

An interactive 3D view of the drone's lawnmower flight path. Waypoints are color-coded by sequence, with vertical drop-lines showing altitude and a green ground plane marking the scan area boundary.

In [24]:
# ── Visualization 1: 3D Flight Trajectory ──
# Renders the full lawnmower flight path in 3D space.
# - X and Y axes show the horizontal scan area (meters)
# - Z axis shows the flight altitude (constant at dataset_spec.height)
# - Waypoints are color-coded by sequence order (Turbo colorscale)
# - Green boundary on the ground plane marks the scan area
# - Vertical drop-lines connect waypoints to the ground for depth perception
# The plot is interactive: rotate, zoom, and hover for per-waypoint details.

fig = plot_3d_trajectory(computed_plan, dataset_spec)
fig.show()

## 2. Camera Footprint Overlay

Overlays the camera's ground footprint rectangles onto the scan area. This shows how individual image captures tile together with the specified overlap and sidelap to provide full coverage.

In [25]:
# ── Visualization 2: Camera Footprint Overlay ──
# Overlays semi-transparent rectangles representing each camera's ground footprint.
# Each rectangle is sized to the image footprint at the current flight height:
#   footprint_x = (num_pixels_x / fx) * height  ≈ 50.6m
#   footprint_y = (num_pixels_y / fy) * height  ≈ 37.9m
# The dense overlap of rectangles shows how 70% overlap/sidelap creates
# heavy redundancy—each ground point is captured by multiple images.
# This redundancy is essential for photogrammetric 3D reconstruction.

fig = plot_footprint_overlay(camera_x10, dataset_spec, computed_plan)
fig.show()

## 3. Parameter Sensitivity Dashboard

A 2×2 dashboard analyzing how key parameters affect the flight plan:
- **Waypoints vs Overlap**: Shows exponential growth as overlap increases
- **Max Speed vs Height**: Linear relationship across exposure times
- **Footprint Area vs Height**: Quadratic growth — doubling height quadruples area
- **GSD vs Height**: Linear degradation of resolution with altitude

In [26]:
# ── Visualization 3: Parameter Sensitivity Dashboard ──
# A 2×2 subplot grid analyzing how key parameters affect flight planning:
#
# Top-Left: Waypoints vs Overlap (at different sidelap values)
#   → Shows exponential growth in mission complexity as overlap increases.
#   → At 90% overlap, the number of waypoints can exceed 600.
#
# Top-Right: Max Blur-Free Speed vs Height (at different exposure times)
#   → Speed = GSD / exposure_time, and GSD = height / focal_length.
#   → So speed ∝ height — flying higher allows faster flight.
#
# Bottom-Left: Footprint Area vs Height
#   → Area = footprint_x × footprint_y, both ∝ height, so area ∝ height².
#   → Quadratic growth means doubling altitude quadruples coverage per image.
#
# Bottom-Right: GSD vs Height
#   → GSD ∝ height — resolution degrades linearly with altitude.
#   → Reference lines at 1cm, 2cm, 5cm show common quality thresholds.

fig = plot_parameter_sensitivity(camera_x10, dataset_spec)
fig.show()

## 4. Overlap × Sidelap Heatmap

Two heatmaps showing how the overlap–sidelap combination affects:
1. **Total waypoints** (mission complexity)
2. **Total flight distance** (time and energy)

High overlap and high sidelap in the top-right corners show the most demanding missions.

In [27]:
# ── Visualization 4: Overlap × Sidelap Heatmap ──
# Generates a grid of flight plans for every combination of overlap (30%-90%)
# and sidelap (30%-90%), computing two metrics for each:
#   1. Total waypoints — measures mission complexity and data volume.
#   2. Total flight distance — measures energy consumption and time.
#
# Key insight: the bottom-left corner (low overlap, low sidelap) produces
# the simplest missions, while the top-right corner (high/high) can require
# 10-20× more waypoints and flight distance.
# This helps pilots choose the minimum overlap/sidelap that meets their
# reconstruction quality requirements without wasting flight time.

fig = plot_overlap_sidelap_heatmap(camera_x10, dataset_spec)
fig.show()

## 5. Multi-Scenario Comparison

Side-by-side bar charts comparing 6 different flight scenarios across 6 metrics: waypoints, flight distance, max speed, GSD, footprint area, and estimated time.

In [28]:
# ── Visualization 5: Multi-Scenario Flight Plan Comparison ──
# Define 6 different mission configurations to compare trade-offs:
#   - Nominal:       our baseline (70%/70%, 30.48m, 2ms)
#   - Low Overlap:   50%/50% — faster coverage, less reconstruction quality
#   - High Overlap:  85%/85% — best for dense 3D models, but very slow
#   - High Altitude: 60m — larger footprint, fewer photos, worse GSD
#   - Low Altitude:  15m — tiny footprint, many photos, excellent GSD
#   - Slow Exposure: 10ms — low-light scenario, very slow flight speed
#
# The bar charts compare: waypoints, distance, speed, GSD, footprint, and time.
# This gives a holistic view of how each parameter affects the mission.

scenarios = [
    ('Nominal\n70%/70%', DatasetSpec(0.7, 0.7, 30.48, 150, 150, 2)),
    ('Low Overlap\n50%/50%', DatasetSpec(0.5, 0.5, 30.48, 150, 150, 2)),
    ('High Overlap\n85%/85%', DatasetSpec(0.85, 0.85, 30.48, 150, 150, 2)),
    ('High Alt\n60m', DatasetSpec(0.7, 0.7, 60, 150, 150, 2)),
    ('Low Alt\n15m', DatasetSpec(0.7, 0.7, 15, 150, 150, 2)),
    ('Slow Exp\n10ms', DatasetSpec(0.7, 0.7, 30.48, 150, 150, 10)),
]
fig = plot_scenario_comparison(camera_x10, scenarios)
fig.show()

## 6. Camera Tilt Angle Analysis

Analysis of how the camera tilt angle (from nadir 90° to oblique 45°) affects:
- Footprint dimensions (exponential growth at low angles)
- Inter-image distance
- Footprint area
- Required waypoints (fewer at oblique angles, but with perspective distortion)

In [29]:
# ── Visualization 6: Camera Tilt Angle Impact Analysis ──
# Sweeps the camera gimbal angle from 45° (oblique) to 90° (nadir/straight down)
# and shows its effect on four quantities:
#
# 1. Footprint Dimensions: as the camera tilts away from nadir, the ground
#    footprint stretches exponentially along the tilt axis (X), while the
#    perpendicular axis (Y) changes only moderately.
#
# 2. Inter-Image Distance: follows the footprint trend — larger footprint
#    means photos can be spaced farther apart for the same overlap.
#
# 3. Footprint Area: grows rapidly at shallow angles.
#
# 4. Waypoints: fewer waypoints needed at oblique angles because each image
#    covers more ground, but the imagery suffers from perspective distortion.
#
# The dashed vertical line marks nadir (90°) as the reference.

fig = plot_angle_analysis(camera_x10, dataset_spec)
fig.show()

## 7. Flight Efficiency Metrics

Gauge dashboard showing key performance indicators:
- **Max Speed**: Limited by GSD and exposure time
- **GSD**: Ground sampling distance (image quality)
- **Est. Time**: Total flight duration estimate
- **Efficiency**: Ratio of useful coverage to total coverage

In [30]:
# ── Visualization 7: Flight Efficiency Metrics (Gauge Dashboard) ──
# Displays four key performance indicators as gauge charts:
#
# 1. Max Speed (m/s): the maximum speed the drone can fly during photo capture
#    without exceeding 1px of motion blur. Computed as GSD / exposure_time.
#    Green zone (>12 m/s) = fast, Red zone (<5 m/s) = slow.
#
# 2. GSD (cm/px): ground sampling distance — how much real-world ground each
#    pixel represents. Lower = better quality. Green (<1cm) = excellent.
#
# 3. Est. Time (sec): total estimated flight time = total_path_distance / speed.
#    Green (<60s) = quick mission, Red (>180s) = long mission.
#
# 4. Efficiency (%): ratio of scan_area to total_coverage (num_images × footprint).
#    High overlap means each point is photographed many times, so efficiency is low.
#    This is expected — redundancy is intentional for photogrammetry.

fig = plot_efficiency_gauge(camera_x10, dataset_spec, computed_plan)
fig.show()